# E1 + E2: response cache

Samples K=4 responses per prompt from the reference model, scores r* and the sentiment axis, caches reference log-probs. ~2-3 T4-hours for 5k prompts. Output: `results/cache/`. Save this notebook's output as a Kaggle dataset; every later notebook imports it.

Repo: https://github.com/Arubad/dpo-misspecification-study.git  
Kaggle: Accelerator GPU T4 x2, Internet On. Results land in `/kaggle/working/results`.

In [ ]:
# --- setup (identical in every notebook) -------------------------------------------------
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
os.chdir(WORK)
if not os.path.exists('dpo-misspecification-study'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Arubad/dpo-misspecification-study.git'], check=True)
os.chdir('dpo-misspecification-study')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
os.environ['RESULTS_ROOT'] = os.path.join(WORK, 'results')
# To reuse caches from an earlier notebook: Add Input -> that notebook's output, then set:
PREV = ''   # e.g. '/kaggle/input/dpo-e1e2-cache/results'
if PREV:
    os.environ['PREV_RESULTS'] = PREV
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


In [ ]:
!python -m src.run build --config configs/cache_0p5b.yaml

In [ ]:
!python -m src.run epsilon --config configs/epsilon_sweep_0p5b.yaml

In [ ]:
import pandas as pd, json
df = pd.read_parquet(os.path.join(os.environ['RESULTS_ROOT'],'cache','responses.parquet'))
print(df.groupby('split').size()); print(df[['n_tokens','rstar_z','f_z','ref_logp']].describe())
print(json.load(open(os.path.join(os.environ['RESULTS_ROOT'],'cache','epsilon_by_minority_mass.json'))))
# W3 artifact: 20 labelled pairs and the r* histogram
from src.train.pairs import build_pairs
import numpy as np
df['row_id']=np.arange(len(df))
pairs = build_pairs(df, {'reward_scale':2.0,'axis_weight':1.0,'gamma':0.0,'mixture':{'K':2,'minority_mass':0.3},'label_flip_eta':0.0}, seed=0)
print(pairs[['prompt_text','chosen','rejected','p_exact','len_w','len_l']].head(20).to_string())
import matplotlib.pyplot as plt; plt.hist(df['rstar_z'], bins=50); plt.title('r* (z-scored)'); plt.show()

In [ ]:
# keep the results as this notebook's output (attach as a dataset to the next notebook)
!du -sh $RESULTS_ROOT && ls $RESULTS_ROOT